# 🎯 Mastering Graders for Reinforcement Fine-Tuning

*A hands-on guide to designing the "brain" behind RFT with Azure OpenAI*

## Reinforcement Learning: The Paradigm Behind Modern LLM Training

### From Games to Language Models

Reinforcement Learning (RL) is a paradigm where an agent learns by interacting with an environment and receiving feedback. Unlike supervised learning where you provide correct answers, RL lets the agent explore and discover what works through trial and reward.

The breakthrough came when DeepMind used RL to master Atari games and Go, domains where the "right move" isn't obvious but success is measurable. The same principle applies to language: we can't always specify the perfect response, but we can recognize quality when we see it.

### RLHF: How LLMs Learn Human Values

Modern LLMs like GPT-4 and Claude aren't just trained on text prediction. After pre-training, they go through **Reinforcement Learning from Human Feedback (RLHF)** which is a process that aligns them with human preferences.

The process works like this: humans compare pairs of model responses and indicate which one is better. These preferences train a *reward model* that learns to predict human judgment. Then, RL fine-tunes the LLM to maximize this reward signal. The result? Models that aren't just fluent, they're helpful, harmless, and honest.

RLHF is why ChatGPT feels different from raw GPT-3. It's why Claude refuses harmful requests politely rather than blindly completing them. The reward signal shaped the model's behavior.

### Large Reasoning Models: RL at the Core

The latest generation of models (o1/o3/o4, GPT-5, DeepSeek-R1...) take this further. These **Large Reasoning Models (LRMs)** use RL not just for alignment, but as their primary training paradigm for complex reasoning.

Instead of learning to imitate human-written solutions, they learn to *think* through problems. The reward signal evaluates the quality of their reasoning process and final answers. This is why SOTA LRM can solve competition math problems and code challenges that stumped earlier models, it learned reasoning strategies and not just patterns.

### RFT: Bringing RL to Your Use Case

**Reinforcement Fine-Tuning (RFT)** lets you apply this same paradigm to your specific domain. Instead of relying on OpenAI's general reward model, you define your own scoring function — a **grader** — that encodes what "good" means for your task.

Want a model that selects the right API tools? Your grader scores tool selection accuracy. Want a model that writes compliant legal summaries? Your grader checks for required clauses. Want a model that diagnoses issues from logs? Your grader validates against known solutions.

The grader is the bridge between RL theory and practical fine-tuning. It's where your domain expertise becomes training signal.

### Why the Grader is Everything

Here's the key insight: **your model can only become as good as your grader can measure**.

If your grader has blind spots, the model will exploit them. If your grader rewards shortcuts, the model will take them. If your grader can't distinguish great from good, the model won't learn the difference.

This is why grader design is the most important skill in RFT. The training algorithm is fixed. The base model is given. But the grader? That's where you encode your expertise, anticipate failure modes, and ultimately determine whether your fine-tuning succeeds or fails.

Let me show you what happens when grader design goes wrong.

## The Story of a Failed Fine-Tuning Job

Let me tell you about a $2,000 mistake.

A team wanted to fine-tune a model to select the right tools for customer service tasks. Simple enough: given "Send the Q3 report to Marie", the model should pick `search_contacts`, then `generate_report`, then `send_email`. In that order.

They wrote a grader that checked if the expected tools appeared in the response. The training ran for hours. The validation scores looked great — 0.95 average! They deployed the model, feeling proud.

Then they tested it.

For *every single request*, the model responded with something like:

> "For this task, I recommend considering: search_contacts, get_user_profile, search_documents, generate_report, send_email, create_calendar_event, get_calendar..."

It had learned to **list every possible tool** because that guaranteed a perfect score. The grader only checked if the right tools were *mentioned* — it never penalized mentioning wrong ones.

The model found a shortcut. It hacked the reward.

**This tutorial exists so you don't make that mistake.**

---

## What You'll Learn

By the end of this notebook, you'll understand how to design graders that actually teach what you want. We'll cover the mechanics of how graders drive learning, explore each grader type Azure offers, and most importantly — learn to anticipate how a model might try to game your scoring system.

We'll use a practical example throughout: building an enterprise assistant that selects the right tools for business tasks. No medical data, no complex dependencies — just a clean, transferable pattern you can adapt to your own use cases.

---

## Part 1: Understanding the Grader's Role

### How RFT Actually Works

Think of Reinforcement Fine-Tuning like training a new employee through feedback rather than scripts.

With **Supervised Fine-Tuning (SFT)**, you hand the employee a manual: "When a customer asks X, say Y." They memorize it. It works, but they can't handle anything that's not in the manual.

With **Reinforcement Fine-Tuning (RFT)**, you let them try things. After each attempt, you give them a score. "That response was a 7 out of 10, good tool selection, but you forgot to check the calendar first." Over time, they learn *principles*, not scripts.

The grader is your scoring system. It's the expert sitting next to the trainee, evaluating every attempt.

Here's what happens during training:

```
┌─────────────────────────────────────────────────────────────────────┐
│                        RFT Training Loop                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ┌──────────────┐      ┌──────────────────────┐                    │
│   │    Prompt    │      │   Model generates    │                    │
│   │  "Send the   │ ───► │   MULTIPLE candidate │                    │
│   │   Q3 report  │      │      responses       │                    │
│   │  to Marie"   │      │                      │                    │
│   └──────────────┘      └──────────────────────┘                    │
│                                   │                                 │
│                                   ▼                                 │
│                         ┌──────────────────┐                        │
│                         │     GRADER       │                        │
│                         │  scores each one │                        │
│                         │   0.3, 0.7, 0.9  │                        │
│                         └──────────────────┘                        │
│                                   │                                 │
│                                   ▼                                 │
│                         ┌──────────────────┐                        │
│                         │  Model learns:   │                        │
│                         │ "Do more of what │                        │
│                         │   scored 0.9"    │                        │
│                         └──────────────────┘                        │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

The grader doesn't just validate — it *teaches*. Whatever it rewards, the model will learn to produce.

### The Sweet Spot Principle

Before you start training, your base model needs to score in a specific range on your grader. Too low, and there's nothing to learn from. Too high, and there's nothing left to improve.

```
Score:  0.0 ──────── 0.3 ════════════ 0.7 ──────── 1.0
              ❌            ✅ Sweet Spot           ❌
           Too hard        Room to improve      Too easy
```

If your base model scores 0.1, it's essentially guessing randomly — there's no "good" behavior to reinforce. If it scores 0.95, you're paying for training that won't improve anything.

The goal of this tutorial is to help you design graders that land in that sweet spot — challenging enough to be useful, but fair enough to provide signal.

---

## Part 2: Setting Up Our Experiment

Let's set up the environment and define our use case. We'll build a grader for an enterprise assistant that selects tools for business tasks.

In [21]:
# Cell 01 - Installation
#!pip install openai python-dotenv azure-ai-evaluation --quiet

#print("✅ Ready to go!")

In [22]:
# Cell 02 - Imports and Azure Client (using Responses API)
import os
import json
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

# Load your Azure credentials from .env file
load_dotenv()

# Use OpenAI client with base_url for Azure OpenAI Responses API
# This pattern is recommended for the Responses API on Azure
client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=f"{os.getenv('AZURE_OPENAI_ENDPOINT')}/openai/v1/"
)

MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT", "o4-mini")

print(f"🔗 Connected to Azure OpenAI (Responses API)")
print(f"📦 Using model: {MODEL}")

🔗 Connected to Azure OpenAI (Responses API)
📦 Using model: o4-mini


### Our Use Case: Enterprise Tool Selection

Imagine an AI assistant that helps employees get things done. When someone says "Send the Q3 report to Marie", the assistant needs to figure out which tools to use and in what order.

Here's our toolkit:

In [23]:
# Cell 03 - Available Tools
TOOLS = {
    "search_contacts": "Find a person's email or profile by name",
    "get_user_profile": "Get the current user's own profile",
    "search_documents": "Search company documents by keyword",
    "generate_report": "Generate a business report (Q1, Q2, etc.)",
    "send_email": "Send an email to a recipient",
    "get_calendar": "Check someone's calendar availability",
    "create_calendar_event": "Schedule a meeting"
}

# Display them nicely
print("🔧 Available Tools\n")
for name, description in TOOLS.items():
    print(f"   {name}")
    print(f"   └─ {description}\n")

🔧 Available Tools

   search_contacts
   └─ Find a person's email or profile by name

   get_user_profile
   └─ Get the current user's own profile

   search_documents
   └─ Search company documents by keyword

   generate_report
   └─ Generate a business report (Q1, Q2, etc.)

   send_email
   └─ Send an email to a recipient

   get_calendar
   └─ Check someone's calendar availability

   create_calendar_event
   └─ Schedule a meeting



And here are some example tasks with their expected tool sequences. Notice how order matters — you can't send an email to Marie before you know her email address.

In [24]:
# Cell 04 - Evaluation Data
EVALUATION_DATA = [
    {
        "task": "Send the Q3 report to Marie",
        "expected_tools": ["search_contacts", "generate_report", "send_email"],
        "reasoning": "First find Marie's email, then create the report, then send it"
    },
    {
        "task": "Schedule a meeting with John tomorrow at 2pm",
        "expected_tools": ["search_contacts", "get_calendar", "create_calendar_event"],
        "reasoning": "Find John, check his availability, then book the meeting"
    },
    {
        "task": "Find all documents about Project Alpha",
        "expected_tools": ["search_documents"],
        "reasoning": "Simple search, no other tools needed"
    },
    {
        "task": "What meetings do I have on Monday?",
        "expected_tools": ["get_user_profile", "get_calendar"],
        "reasoning": "Get own profile for calendar access, then check calendar"
    },
    {
        "task": "Email the sales team our Q1 results",
        "expected_tools": ["search_contacts", "generate_report", "send_email"],
        "reasoning": "Find team emails, generate report, send to all"
    }
]

print(f"📋 Loaded {len(EVALUATION_DATA)} evaluation scenarios")

📋 Loaded 5 evaluation scenarios


---

## Part 3: The Grader Types

Azure OpenAI offers four types of graders, each suited to different situations. Let's explore them one by one, starting with the simplest.

### 3.1 String Check — The Simple Comparator

A string check grader compares the model's output against a reference string. It's fast, deterministic, and returns either 0 or 1.

Think of it like checking if a student's answer matches the answer key exactly.

In [25]:
# Cell 05 - String Check Config
# Here's what a string_check grader looks like in Azure's format
string_check_config = {
    "type": "string_check",
    "name": "exact_match",
    "input": "{{sample.output_text}}",      # The model's response
    "reference": "{{item.expected_output}}", # What we want
    "operation": "eq"                        # Must be exactly equal
}

print("String Check Grader Configuration:")
print(json.dumps(string_check_config, indent=2))

String Check Grader Configuration:
{
  "type": "string_check",
  "name": "exact_match",
  "input": "{{sample.output_text}}",
  "reference": "{{item.expected_output}}",
  "operation": "eq"
}


The available operations are:

| Operation | Behavior | Case-sensitive? |
|-----------|----------|----------------|
| `eq` | Input must match reference exactly | Yes |
| `ne` | Input must NOT match reference | Yes |
| `like` | Reference must appear somewhere in input | Yes |
| `ilike` | Same as `like` but case-insensitive | No |

Let's see how it behaves:

In [26]:
# Cell 06 - String Check Implementation
def string_check(output: str, reference: str, operation: str = "eq") -> float:
    """Simulates Azure's string_check grader locally."""
    if operation == "eq":
        return 1.0 if output == reference else 0.0
    elif operation == "ne":
        return 1.0 if output != reference else 0.0
    elif operation == "like":
        return 1.0 if reference in output else 0.0
    elif operation == "ilike":
        return 1.0 if reference.lower() in output.lower() else 0.0
    return 0.0

# Test it out
print("Testing string_check grader:\n")

tests = [
    ("send_email", "send_email", "eq", "Exact match"),
    ("Send_Email", "send_email", "eq", "Case mismatch with eq"),
    ("Send_Email", "send_email", "ilike", "Case mismatch with ilike"),
    ("Use send_email for this", "send_email", "like", "Substring match"),
]

for output, ref, op, description in tests:
    score = string_check(output, ref, op)
    emoji = "✅" if score == 1.0 else "❌"
    print(f"{emoji} {description}")
    print(f"   '{output}' vs '{ref}' ({op}) → {score}\n")

Testing string_check grader:

✅ Exact match
   'send_email' vs 'send_email' (eq) → 1.0

❌ Case mismatch with eq
   'Send_Email' vs 'send_email' (eq) → 0.0

✅ Case mismatch with ilike
   'Send_Email' vs 'send_email' (ilike) → 1.0

✅ Substring match
   'Use send_email for this' vs 'send_email' (like) → 1.0



<div style="border-left: 4px solid #ffc107; padding: 12px 16px; margin: 16px 0; background: #fffbeb; color: #000;">
    <strong>⚠️ The "Like" Trap</strong><br>
    Using <code>like</code> or <code>ilike</code> seems convenient, but it's dangerous. The check only verifies the reference appears <em>somewhere</em> in the output — it doesn't care about context.
</div>

In [27]:
# Cell 07 - Like Trap Demo
# Here's the danger:
dangerous_output = "Do NOT use send_email — this requires human approval"
reference = "send_email"

score = string_check(dangerous_output, reference, "like")
print(f"Output: '{dangerous_output}'")
print(f"Reference: '{reference}'")
print(f"Score with 'like': {score} 😱")
print("\nThe model said NOT to use it, but got full credit!")

Output: 'Do NOT use send_email — this requires human approval'
Reference: 'send_email'
Score with 'like': 1.0 😱

The model said NOT to use it, but got full credit!


**When to use string_check:** Classification tasks with fixed labels, yes/no answers, or when you need the output to match a specific format exactly. Avoid `like` for anything where context matters.

---

### 3.2 Text Similarity — The Fuzzy Matcher

Text similarity graders measure how close the model's output is to a reference, returning a score between 0 and 1. They're useful when you want to reward "close enough" answers.

Azure supports several metrics:

| Metric | What it measures |
|--------|------------------|
| `fuzzy_match` | Character-level similarity (using rapidfuzz) |
| `bleu` | N-gram overlap (common in translation) |
| `gleu` | Google's variant of BLEU |
| `meteor` | Semantic similarity with synonyms |
| `rouge_1` to `rouge_l` | Recall-oriented overlap |

In [28]:
# Cell 08 - Text Similarity Config
text_similarity_config = {
    "type": "text_similarity",
    "name": "fuzzy_match",
    "input": "{{sample.output_text}}",
    "reference": "{{item.expected_output}}",
    "evaluation_metric": "fuzzy_match"
}

print("Text Similarity Grader Configuration:")
print(json.dumps(text_similarity_config, indent=2))

Text Similarity Grader Configuration:
{
  "type": "text_similarity",
  "name": "fuzzy_match",
  "input": "{{sample.output_text}}",
  "reference": "{{item.expected_output}}",
  "evaluation_metric": "fuzzy_match"
}


In [29]:
# Cell 09 - Fuzzy Match Implementation
from difflib import SequenceMatcher

def fuzzy_match(output: str, reference: str) -> float:
    """Simulates Azure's fuzzy_match metric."""
    return SequenceMatcher(None, output.lower(), reference.lower()).ratio()

# Test with tool sequences
reference = "search_contacts, generate_report, send_email"

test_cases = [
    ("search_contacts, generate_report, send_email", "Perfect match"),
    ("search_contacts, send_email, generate_report", "Wrong order"),
    ("search_contacts, generate_report", "Missing one"),
    ("search_documents, generate_report, send_email", "Wrong first tool"),
]

print(f"Reference: {reference}\n")
print("-" * 60)

for output, description in test_cases:
    score = fuzzy_match(output, reference)
    bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
    print(f"\n{description}")
    print(f"Output: {output}")
    print(f"Score:  [{bar}] {score:.2f}")

Reference: search_contacts, generate_report, send_email

------------------------------------------------------------

Perfect match
Output: search_contacts, generate_report, send_email
Score:  [████████████████████] 1.00

Wrong order
Output: search_contacts, send_email, generate_report
Score:  [██████████████░░░░░░] 0.73

Missing one
Output: search_contacts, generate_report
Score:  [████████████████░░░░] 0.84

Wrong first tool
Output: search_documents, generate_report, send_email
Score:  [█████████████████░░░] 0.90


<div style="border-left: 4px solid #ffc107; padding: 12px 16px; margin: 16px 0; background: #fffbeb; color: #000;">
    <strong>⚠️ The Similarity Illusion</strong><br>
    Notice how "search_documents" vs "search_contacts" scores high because the strings are similar? Text similarity doesn't understand that these are <em>completely different tools</em>. It just sees similar characters. For tool selection, this is dangerous — a model could learn to output similar-looking but wrong tool names.
</div>

**When to use text_similarity:** Summarization tasks, translations, or cases where you want to reward outputs that capture the same content with different wording. Not ideal for structured outputs with specific required values.

---

### 3.3 Python Grader — The Flexible Expert ⭐

This is where things get interesting. A Python grader lets you write custom scoring logic that runs in Azure's sandbox. You can check exact tool names, verify ordering, implement partial credit — whatever your use case requires.

This is the grader type you'll use most often for complex tasks.

**Combined with structured output** (which we'll cover in section 3.5), Python graders become especially powerful, you can parse clean JSON instead of hunting through free-form text.

In [30]:
# Cell 10 - Python Grader (F1 + JSON)
# The code that will run in Azure's sandbox
# This grader expects structured JSON output from the model
GRADER_CODE = '''
import json

def grade(sample: dict, item: dict) -> float:
    """
    Scores a tool selection response with structured JSON output.
    
    Expects the model to output JSON like:
    {"reasoning": "...", "tools": ["tool1", "tool2"]}
    
    We evaluate two things:
    1. RECALL - Did the model predict the required tools?
    2. PRECISION - Did it avoid predicting unnecessary tools?
    
    Note: With structured output, ORDER is implicit in the array order,
    so we focus on set-based metrics for simplicity.
    """
    
    # --- Parse inputs safely ---
    try:
        # Parse the model's JSON response
        response_text = sample.get("output_text", "{}")
        response = json.loads(response_text)
        predicted_tools = [t.lower() for t in response.get("tools", [])]
        
        # Parse the reference answer
        ref_data = item.get("reference_answer", "{}")
        if isinstance(ref_data, str):
            ref_data = json.loads(ref_data)
        expected_tools = [t.lower() for t in ref_data.get("expected_tools", [])]
        
        if not expected_tools:
            return 0.0
            
    except (json.JSONDecodeError, AttributeError, TypeError):
        return 0.0
    
    # --- Convert to sets for comparison ---
    predicted_set = set(predicted_tools)
    expected_set = set(expected_tools)
    
    # --- RECALL: What fraction of expected tools were predicted? ---
    if len(expected_set) == 0:
        recall = 0.0
    else:
        correct = len(predicted_set & expected_set)
        recall = correct / len(expected_set)
    
    # --- PRECISION: What fraction of predicted tools were correct? ---
    if len(predicted_set) == 0:
        precision = 0.0
    else:
        correct = len(predicted_set & expected_set)
        precision = correct / len(predicted_set)
    
    # --- F1 SCORE: Harmonic mean of precision and recall ---
    if precision + recall == 0:
        return 0.0
    
    f1 = 2 * (precision * recall) / (precision + recall)
    
    return min(max(f1, 0.0), 1.0)
'''

# Execute it locally so we can test
exec(GRADER_CODE)

print("✅ Grader loaded and ready for testing")

✅ Grader loaded and ready for testing


Let's understand what this grader does:

**Recall** measures what fraction of the expected tools were predicted. Missing a tool means the task can't be completed — if you don't call `search_contacts`, you'll never find Marie's email.

**Precision** measures what fraction of predicted tools were actually needed. This prevents the "list everything" hack we saw earlier.

**F1 Score** is the harmonic mean of precision and recall, giving us a single balanced metric.

With structured output, we don't need to:
- Search for tool names in free text
- Handle variations like "search_contacts" vs "search contacts"
- Check for tool order manually (the array preserves order)

Now let's test it:

In [31]:
# Cell 11 - Grader Tests
def test_grader(response_json: dict, expected_tools: list) -> dict:
    """Helper to test our grader with nice output."""
    sample = {"output_text": json.dumps(response_json)}
    item = {"reference_answer": json.dumps({"expected_tools": expected_tools})}
    score = grade(sample, item)
    return {"tools": response_json.get("tools", []), "score": score}

expected = ["search_contacts", "generate_report", "send_email"]

test_cases = [
    (
        {"reasoning": "Find Marie, create report, send it", "tools": ["search_contacts", "generate_report", "send_email"]},
        "Perfect response"
    ),
    (
        {"reasoning": "Wrong order but all tools", "tools": ["send_email", "search_contacts", "generate_report"]},
        "All tools (order doesn't affect F1)"
    ),
    (
        {"reasoning": "Missing one tool", "tools": ["search_contacts", "send_email"]},
        "Missing generate_report"
    ),
    (
        {"reasoning": "List everything", "tools": ["search_contacts", "get_user_profile", "search_documents", "generate_report", "send_email", "get_calendar", "create_calendar_event"]},
        "Lists ALL tools (the hack!)"
    ),
    (
        {"reasoning": "No tools identified", "tools": []},
        "Empty tools array"
    ),
]

print(f"Expected tools: {expected}\n")
print("=" * 70)

for response_json, description in test_cases:
    result = test_grader(response_json, expected)
    score = result["score"]
    
    # Visual bar
    bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
    emoji = "✅" if score > 0.8 else "⚠️" if score > 0.5 else "❌"
    
    print(f"\n{emoji} {description}")
    print(f"   Tools: {result['tools']}")
    print(f"   [{bar}] {score:.2f}")

Expected tools: ['search_contacts', 'generate_report', 'send_email']


✅ Perfect response
   Tools: ['search_contacts', 'generate_report', 'send_email']
   [████████████████████] 1.00

✅ All tools (order doesn't affect F1)
   Tools: ['send_email', 'search_contacts', 'generate_report']
   [████████████████████] 1.00

⚠️ Missing generate_report
   Tools: ['search_contacts', 'send_email']
   [████████████████░░░░] 0.80

⚠️ Lists ALL tools (the hack!)
   Tools: ['search_contacts', 'get_user_profile', 'search_documents', 'generate_report', 'send_email', 'get_calendar', 'create_calendar_event']
   [████████████░░░░░░░░] 0.60

❌ Empty tools array
   Tools: []
   [░░░░░░░░░░░░░░░░░░░░] 0.00


Notice how the "Lists ALL tools" response gets penalized? That's the precision component of the F1 score doing its job. The hack that worked against our naive (recall-only) grader now scores poorly.

<div style="border-left: 4px solid #28a745; padding: 12px 16px; margin: 16px 0; background: #f0fff4; color: #000;">
    <strong>💡 Key Insight</strong><br>
    A good grader rewards what you want AND penalizes what you don't want. The F1 score naturally balances both concerns: recall ensures you find the right tools, precision ensures you don't include wrong ones.
</div>

---

### 3.4 Model Grader — The LLM Judge

Sometimes you need to evaluate qualities that are hard to express in code: Is the response helpful? Is the tone appropriate? Does the explanation make sense?

A model grader uses another LLM (like GPT-4o) to evaluate the response. It's powerful but comes with tradeoffs: it's slower, more expensive, and can be inconsistent.

Azure supports these models as graders: `gpt-4o-2024-08-06` and `o3-mini-2025-01-31`.

In [32]:
# Cell 12 - LLM Judge Prompt
JUDGE_PROMPT = """
You are evaluating an AI assistant's tool selection for a business task.

## The Task
User request: {{item.task}}
Expected tools (in order): {{item.expected_tools}}

## The Response
{{sample.output_text}}

## Evaluation Criteria
1. Did the assistant identify the correct tools?
2. Are they in a logical order (dependencies respected)?
3. Is the reasoning clear and sound?

## Scoring Guide
- 1.0: Perfect tool selection, correct order, clear reasoning
- 0.75: Correct tools, minor order or explanation issues
- 0.5: Missing one tool or significant order problems
- 0.25: Multiple errors but shows understanding
- 0.0: Completely wrong or irrelevant

Respond with ONLY a number (0.0, 0.25, 0.5, 0.75, or 1.0).
"""

model_grader_config = {
    "type": "score_model",
    "name": "llm_judge",
    "model": "gpt-4o-2024-08-06",
    "input": [
        {
            "role": "user",
            "type": "message",
            "content": JUDGE_PROMPT
        }
    ]
}

print("Model Grader Configuration:")
print(f"  Model: {model_grader_config['model']}")
print(f"  Prompt length: {len(JUDGE_PROMPT)} characters")

Model Grader Configuration:
  Model: gpt-4o-2024-08-06
  Prompt length: 710 characters


<div style="border-left: 4px solid #ffc107; padding: 12px 16px; margin: 16px 0; background: #fffbeb; color: #000;">
    <strong>⚠️ LLM Judges Have Biases</strong><br>
    LLMs tend to prefer longer, more verbose responses. They can also be inconsistent — the same response might get different scores on different runs.<br><br>
    <strong>Mitigations:</strong> Use discrete scores (0, 0.25, 0.5, 0.75, 1) instead of continuous. Include examples of good and bad responses in your prompt. Test the judge multiple times on the same inputs to check consistency.
</div>

**When to use model graders:** Open-ended responses, tone/style evaluation, or when you need nuanced judgment that's hard to express in code. Use sparingly due to cost and latency.

---

### 3.5 Response Format — Structured Output for Reliable Parsing

Before we discuss multi-graders, let's address a fundamental question: **what format should the model output?**

By default, models produce free-form text. This creates parsing challenges — your grader must search for tool names in a sea of prose, handle variations like "search_contacts" vs "search contacts" vs "SearchContacts", and deal with false positives when tools are mentioned in context ("Do NOT use send_email").

Azure OpenAI RFT supports **structured output** via the `response_format` parameter. Instead of free-form text, you can force the model to produce valid JSON matching a schema you define. This eliminates parsing ambiguity entirely.

<div style="border-left: 4px solid #28a745; padding: 12px 16px; margin: 16px 0; background: #f0fff4; color: #000;">
    <strong>💡 Best Practice</strong><br>
    Use structured output whenever your task has a well-defined output format. It makes grading simpler, more reliable, and eliminates entire categories of reward hacking.
</div>

In [33]:
# Cell 13 - Response Format Schema
# Note: The schema format differs between APIs:
# - Fine-tuning API uses: response_format={"type": "json_schema", "json_schema": {...}}
# - Responses API uses: text={"format": {"type": "json_schema", "name": "...", "schema": {...}}}

# Training format (for RFT jobs)
RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "tool_selection",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "reasoning": {
                    "type": "string",
                    "description": "Step-by-step explanation of tool selection"
                },
                "tools": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Ordered list of tool names to use"
                }
            },
            "required": ["reasoning", "tools"],
            "additionalProperties": False
        }
    }
}

# For Responses API: must include "type": "json_schema" at the format level
RESPONSES_API_FORMAT = {
    "type": "json_schema",
    "name": RESPONSE_FORMAT["json_schema"]["name"],
    "strict": RESPONSE_FORMAT["json_schema"]["strict"],
    "schema": RESPONSE_FORMAT["json_schema"]["schema"]
}

print("Response Format Schema (Training format):")
print(json.dumps(RESPONSE_FORMAT, indent=2))
print("\n" + "-" * 40)
print("\nResponses API format (text.format):")
print(json.dumps(RESPONSES_API_FORMAT, indent=2))

Response Format Schema (Training format):
{
  "type": "json_schema",
  "json_schema": {
    "name": "tool_selection",
    "strict": true,
    "schema": {
      "type": "object",
      "properties": {
        "reasoning": {
          "type": "string",
          "description": "Step-by-step explanation of tool selection"
        },
        "tools": {
          "type": "array",
          "items": {
            "type": "string"
          },
          "description": "Ordered list of tool names to use"
        }
      },
      "required": [
        "reasoning",
        "tools"
      ],
      "additionalProperties": false
    }
  }
}

----------------------------------------

Responses API format (text.format):
{
  "type": "json_schema",
  "name": "tool_selection",
  "strict": true,
  "schema": {
    "type": "object",
    "properties": {
      "reasoning": {
        "type": "string",
        "description": "Step-by-step explanation of tool selection"
      },
      "tools": {
        "type": 

---

### 3.6 Multi Grader — Combining Perspectives

A multi grader lets you combine multiple graders with weighted scores. This is useful when you want to evaluate different aspects independently and then aggregate them.

<div style="border-left: 4px solid #dc3545; padding: 12px 16px; margin: 16px 0; background: #fff5f5; color: #000;">
    <strong>🚫 Azure Limitation</strong><br>
    Multi graders <strong>cannot contain model graders</strong> as sub-graders. You can only combine <code>string_check</code>, <code>text_similarity</code>, and <code>python</code> graders.
</div>

In [34]:
# Cell 14 - Multi Grader Config
# Example multi-grader combining tool accuracy and format checking
multi_grader_config = {
    "type": "multi",
    "graders": {
        "tool_accuracy": {
            "type": "python",
            "name": "tool_check",
            "source": """import json
def grade(sample, item):
    try:
        response = json.loads(sample.get('output_text', '{}'))
        predicted = set(t.lower() for t in response.get('tools', []))
        ref = json.loads(item.get('reference_answer', '{}'))
        expected = set(t.lower() for t in ref.get('expected_tools', []))
        if not expected:
            return 0.0
        correct = len(predicted & expected)
        recall = correct / len(expected)
        precision = correct / len(predicted) if predicted else 0.0
        return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    except:
        return 0.0
"""
        },
        "has_reasoning": {
            "type": "python",
            "name": "reasoning_check",
            "source": """import json
def grade(sample, item):
    try:
        response = json.loads(sample.get('output_text', '{}'))
        reasoning = response.get('reasoning', '')
        return 1.0 if len(reasoning) > 20 else 0.5 if reasoning else 0.0
    except:
        return 0.0
"""
        }
    },
    "calculate_output": "0.8 * tool_accuracy + 0.2 * has_reasoning"
}

print("Multi Grader Configuration:")
print(f"  Sub-graders: {list(multi_grader_config['graders'].keys())}")
print(f"  Formula: {multi_grader_config['calculate_output']}")

Multi Grader Configuration:
  Sub-graders: ['tool_accuracy', 'has_reasoning']
  Formula: 0.8 * tool_accuracy + 0.2 * has_reasoning


In [35]:
# Cell 15 - System Prompt
SYSTEM_PROMPT = f"""
You are an enterprise assistant that helps users accomplish tasks by selecting the right tools.

Available tools:
{chr(10).join(f'  - {name}: {desc}' for name, desc in TOOLS.items())}

When given a task:
1. Analyze what needs to be done
2. Identify which tools are needed and in what order
3. Provide your reasoning and the ordered list of tools

You must respond with valid JSON matching the required schema.
"""

print("System prompt ready")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

System prompt ready
Length: 728 characters


In [36]:
# Cell 16 - Get Response (Structured Output with Responses API)
def get_response(task: str, debug: bool = False) -> dict:
    """Get a structured JSON response from the model using Responses API."""
    try:
        # Responses API uses:
        # - 'instructions' for system prompt (or include in 'input' with role)
        # - 'input' for user message(s)
        # - 'text.format' for structured output schema
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_PROMPT,
            input=task,
            text={"format": RESPONSES_API_FORMAT}
        )
        if debug:
            print(f"Raw response: {response.output_text[:500] if response.output_text else 'None'}")
        return json.loads(response.output_text)
    except Exception as e:
        print(f"Error for task '{task}': {e}")
        return {"reasoning": f"Error: {e}", "tools": []}

# Test with one example (with debug enabled)
test_task = EVALUATION_DATA[0]["task"]
print(f"Task: {test_task}\n")
test_response = get_response(test_task, debug=True)

print(f"\nParsed Response:")
print(json.dumps(test_response, indent=2))

Task: Send the Q3 report to Marie

Raw response: {"reasoning":"The task requires obtaining the Q3 report, finding Marie’s email address, and then emailing the report to her. First, generate the Q3 report. Next, look up Marie’s contact information. Finally, send an email with the report attached.","tools":["generate_report","search_contacts","send_email"]}

Parsed Response:
{
  "reasoning": "The task requires obtaining the Q3 report, finding Marie\u2019s email address, and then emailing the report to her. First, generate the Q3 report. Next, look up Marie\u2019s contact information. Finally, send an email with the report attached.",
  "tools": [
    "generate_report",
    "search_contacts",
    "send_email"
  ]
}


In [37]:
# Cell 17 - Base Model Evaluation
# Evaluate the base model on all our test cases
print("\n" + "=" * 70)
print("              EVALUATING BASE MODEL (with Structured Output)")
print("=" * 70 + "\n")

scores = []

for data in EVALUATION_DATA:
    task = data["task"]
    expected = data["expected_tools"]
    
    # Get model response (structured JSON)
    response = get_response(task)
    
    # Score it
    sample = {"output_text": json.dumps(response)}
    item = {"reference_answer": json.dumps({"expected_tools": expected})}
    score = grade(sample, item)
    scores.append(score)
    
    # Display
    bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
    emoji = "✅" if score > 0.8 else "⚠️" if score > 0.5 else "❌"
    
    print(f"{emoji} {task}")
    print(f"   Expected: {expected}")
    print(f"   Got:      {response.get('tools', [])}")
    print(f"   Score: [{bar}] {score:.2f}\n")

# Summary
avg_score = sum(scores) / len(scores)
print("=" * 70)
print(f"\n📊 Average Score: {avg_score:.2f}")

if 0.3 <= avg_score <= 0.7:
    print("\n✅ SWEET SPOT! This is a good candidate for RFT.")
    print("   The model has room to improve but isn't completely lost.")
elif avg_score < 0.3:
    print("\n⚠️ Score too low. The model needs more guidance.")
    print("   Consider: simpler task, better prompt, or easier examples.")
else:
    print("\n⚠️ Score too high. The model is already good at this.")
    print("   RFT might not provide significant improvement.")


              EVALUATING BASE MODEL (with Structured Output)

✅ Send the Q3 report to Marie
   Expected: ['search_contacts', 'generate_report', 'send_email']
   Got:      ['search_contacts', 'generate_report', 'send_email']
   Score: [████████████████████] 1.00

✅ Schedule a meeting with John tomorrow at 2pm
   Expected: ['search_contacts', 'get_calendar', 'create_calendar_event']
   Got:      ['search_contacts', 'get_calendar', 'create_calendar_event']
   Score: [████████████████████] 1.00

✅ Find all documents about Project Alpha
   Expected: ['search_documents']
   Got:      ['search_documents']
   Score: [████████████████████] 1.00

⚠️ What meetings do I have on Monday?
   Expected: ['get_user_profile', 'get_calendar']
   Got:      ['get_calendar']
   Score: [█████████████░░░░░░░] 0.67

✅ Email the sales team our Q1 results
   Expected: ['search_contacts', 'generate_report', 'send_email']
   Got:      ['generate_report', 'search_contacts', 'send_email']
   Score: [████████████████

In [38]:
# Cell 18 - Reward Hacking Demo
print("\n" + "=" * 70)
print("           🚨 REWARD HACKING DEMONSTRATION")
print("=" * 70 + "\n")

# A naive grader that only checks recall
def naive_grader(response: dict, expected: list) -> float:
    """Only checks if expected tools are mentioned (no precision penalty)."""
    predicted = response.get("tools", [])
    found = sum(1 for t in expected if t in predicted)
    return found / len(expected) if expected else 0.0

expected = ["search_contacts", "generate_report", "send_email"]

# The "hacked" response - lists all tools
hacked_response = {
    "reasoning": "To be thorough, let's use all available tools",
    "tools": ["search_contacts", "get_user_profile", "search_documents", 
              "generate_report", "send_email", "get_calendar", "create_calendar_event"]
}

# A good response - only the needed tools
good_response = {
    "reasoning": "Find Marie's email, create the Q3 report, then send it",
    "tools": ["search_contacts", "generate_report", "send_email"]
}

print("HACKED RESPONSE (lists all tools):\n")
print(f"  Tools: {hacked_response['tools']}")
print(f"  Naive grader (recall only):  {naive_grader(hacked_response, expected):.2f} 😱")

# Use our robust grader
sample = {"output_text": json.dumps(hacked_response)}
item = {"reference_answer": json.dumps({"expected_tools": expected})}
print(f"  F1 grader (with precision): {grade(sample, item):.2f} ✓")

print("\n" + "-" * 70)
print("\nGOOD RESPONSE (correct tools only):\n")
print(f"  Tools: {good_response['tools']}")
print(f"  Naive grader (recall only):  {naive_grader(good_response, expected):.2f}")

sample = {"output_text": json.dumps(good_response)}
print(f"  F1 grader (with precision): {grade(sample, item):.2f} ✓")

print("\n" + "=" * 70)
print("\n💡 LESSON: The F1 grader penalizes the hack via precision.")
print("   Without precision scoring, the model learns to list all tools.")


           🚨 REWARD HACKING DEMONSTRATION

HACKED RESPONSE (lists all tools):

  Tools: ['search_contacts', 'get_user_profile', 'search_documents', 'generate_report', 'send_email', 'get_calendar', 'create_calendar_event']
  Naive grader (recall only):  1.00 😱
  F1 grader (with precision): 0.60 ✓

----------------------------------------------------------------------

GOOD RESPONSE (correct tools only):

  Tools: ['search_contacts', 'generate_report', 'send_email']
  Naive grader (recall only):  1.00
  F1 grader (with precision): 1.00 ✓


💡 LESSON: The F1 grader penalizes the hack via precision.
   Without precision scoring, the model learns to list all tools.


### Common Hacks and How to Prevent Them

| What the model learns to do | Why it works | How to prevent |
|----------------------------|--------------|----------------|
| List all possible tools | Guarantees recall | Add precision penalty (F1 score) |
| Be extremely verbose | LLM judges prefer detail | Use code grader with structured output |
| ~~Repeat the question back~~ | ~~Increases similarity scores~~ | *Eliminated by structured output* |
| ~~Include "NOT" before wrong answers~~ | ~~Substring matching triggers~~ | *Eliminated by structured output* |

<div style="border-left: 4px solid #28a745; padding: 12px 16px; margin: 16px 0; background: #f0fff4; color: #000;">
    <strong>💡 Structured Output Eliminates Parsing Hacks</strong><br>
    Notice how several classic reward hacks (repeating questions, using "NOT" prefixes) simply don't apply when using structured output. The model must output a valid JSON array of tool names — there's no room for word games or ambiguous phrasing.
</div>

---

## Part 6: Your Pre-Flight Checklist

Before you launch an RFT job, run through this checklist:

**Grader Design**
- ☐ Uses partial credit (not just 0 and 1)
- ☐ Penalizes unwanted behavior (not just rewards correct behavior)
- ☐ Handles edge cases (empty input, malformed JSON)
- ☐ Returns scores between 0.0 and 1.0
- ☐ Tested with adversarial inputs (hacks, edge cases)

**Structured Output**
- ☐ JSON schema defined with required fields
- ☐ Schema uses `strict: true` for guaranteed compliance
- ☐ Grader parses JSON instead of searching text

**Sweet Spot Verification**
- ☐ Base model scores 0.3 - 0.7 on average
- ☐ Scores have variance (not all identical)
- ☐ Both high and low scores present (signal exists)

**Data Quality**
- ☐ Reference answers are correct and consistent
- ☐ Prompts are clear and unambiguous
- ☐ At least 50-100 examples for training

**If Using LLM Judge**
- ☐ Prompt includes scoring examples
- ☐ Uses discrete scores (0, 0.25, 0.5, 0.75, 1.0)
- ☐ Tested for consistency (same input → same output)

In [39]:
# Cell 19 - Production Grader
PRODUCTION_GRADER = '''
import json

def grade(sample: dict, item: dict) -> float:
    """
    Production-ready grader for tool selection tasks with structured output.
    
    Expects JSON input: {"reasoning": "...", "tools": ["tool1", "tool2"]}
    
    Scoring: F1 score (harmonic mean of precision and recall)
      - RECALL: What fraction of expected tools were predicted?
      - PRECISION: What fraction of predicted tools were correct?
    
    Returns: float between 0.0 and 1.0
    """
    
    # ─────────────────────────────────────────────────────────────
    # PARSE INPUTS SAFELY
    # ─────────────────────────────────────────────────────────────
    try:
        # Parse model's JSON response
        response_text = sample.get("output_text", "{}")
        response = json.loads(response_text)
        predicted = [t.lower() for t in response.get("tools", [])]
        
        # Parse reference answer
        ref_data = item.get("reference_answer", "{}")
        if isinstance(ref_data, str):
            ref_data = json.loads(ref_data)
        expected = [t.lower() for t in ref_data.get("expected_tools", [])]
        
        if not expected:
            return 0.0
            
    except (json.JSONDecodeError, AttributeError, TypeError):
        return 0.0
    
    # ─────────────────────────────────────────────────────────────
    # COMPUTE F1 SCORE
    # ─────────────────────────────────────────────────────────────
    predicted_set = set(predicted)
    expected_set = set(expected)
    
    # Recall: fraction of expected tools that were predicted
    correct = len(predicted_set & expected_set)
    recall = correct / len(expected_set) if expected_set else 0.0
    
    # Precision: fraction of predicted tools that were correct
    precision = correct / len(predicted_set) if predicted_set else 0.0
    
    # F1: harmonic mean
    if precision + recall == 0:
        return 0.0
    
    f1 = 2 * (precision * recall) / (precision + recall)
    
    return min(max(f1, 0.0), 1.0)
'''

# The Azure configuration
GRADER_CONFIG = {
    "type": "python",
    "name": "tool_selection_grader",
    "source": PRODUCTION_GRADER
}

print("✅ Production grader ready")
print(f"   Type: {GRADER_CONFIG['type']}")
print(f"   Name: {GRADER_CONFIG['name']}")
print(f"   Code: {len(PRODUCTION_GRADER)} characters")

✅ Production grader ready
   Type: python
   Name: tool_selection_grader
   Code: 1923 characters


---

## Part 7: Putting It All Together

Here's the complete grader, ready for production use:

In [40]:
# Cell 20 - RFT Training Job Config
# ═══════════════════════════════════════════════════════════════════
#  TRAINING JOB - Uncomment when ready
# ═══════════════════════════════════════════════════════════════════

'''
# 1. Prepare your training data
training_data = []
for item in EVALUATION_DATA:
    training_data.append({
        "messages": [
            {"role": "developer", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["task"]}
        ],
        "reference_answer": json.dumps({
            "expected_tools": item["expected_tools"]
        })
    })

# 2. Save to JSONL file
with open("training_data.jsonl", "w") as f:
    for record in training_data:
        f.write(json.dumps(record) + "\n")

# 3. Upload to Azure
training_file = client.files.create(
    file=open("training_data.jsonl", "rb"),
    purpose="fine-tune"
)

# 4. Launch the RFT job with structured output
job = client.fine_tuning.jobs.create(
    model="o4-mini-2025-04-16",
    training_file=training_file.id,
    method={
        "type": "reinforcement",
        "reinforcement": {
            "grader": GRADER_CONFIG,
            # Force structured JSON output during training
            "response_format": RESPONSE_FORMAT
        }
    },
    hyperparameters={
        "n_epochs": 2,
        "reasoning_effort": "medium"
    }
)

print(f"✅ Training job started: {job.id}")
'''

print("💡 Uncomment the code above when you're ready to train.")
print("   Make sure you've tested your grader thoroughly first!")
print("\n📋 Key configuration:")
print(f"   - Grader: Python (F1 score)")
print(f"   - Response format: JSON schema (structured output)")

💡 Uncomment the code above when you're ready to train.
   Make sure you've tested your grader thoroughly first!

📋 Key configuration:
   - Grader: Python (F1 score)
   - Response format: JSON schema (structured output)


---

## Summary: What We Learned

We covered a lot of ground. Here are the key takeaways:

**The grader is the teacher.** Whatever it rewards, the model will learn. Design it carefully.

**Use structured output.** Force the model to produce JSON matching a schema. This eliminates parsing ambiguity and prevents entire categories of reward hacking.

**Test before training.** Validate your grader and check that the base model scores in the sweet spot (0.3-0.7).

**Reward what you want, penalize what you don't.** A grader that only checks for correct behavior will get gamed. Use F1 score to balance recall and precision.

**Match the grader to the task:**

| Task type | Best grader |
|-----------|-------------|
| Exact matches | `string_check` |
| Similar content | `text_similarity` |
| Complex logic with structured output | `python` ⭐ |
| Subjective quality | `score_model` |
| Multiple criteria | `multi` |

**LLM judges are powerful but tricky.** They're slow, expensive, and can be inconsistent. Use them when code can't capture what "good" means, but test for reliability.

---

## 🎉 You're Ready to Build Great Graders

Go forth and fine-tune responsibly.

---

### Resources

- [Azure OpenAI RFT Documentation](https://learn.microsoft.com/azure/ai-services/openai/how-to/reinforcement-fine-tuning)
- [OpenAI Graders Guide](https://platform.openai.com/docs/guides/graders)
- [Azure AI Evaluation SDK](https://learn.microsoft.com/azure/ai-studio/how-to/evaluate-generative-ai-app)